.apply(group_icd9) recorre fila por fila la columna diag_1, y para cada valor llama a la función group_icd9 con ese valor. El resultado de cada llamada se convierte en el valor de la nueva columna diag_1_group.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/diabetic_data_clean.csv')

def group_icd9(code):
    """
    Groups ICD-9 codes into clinical categories.
    Source: standard ICD-9-CM classification
    """
    if pd.isna(code) or code == 'Unknown':
        return 'Unknown'
    
    code = str(code).strip()
    
    # Codes starting with a letter — external causes or outpatient visits
    if code.startswith('V') or code.startswith('E'):
        return 'External'
    
    try:
        num = float(code)
    except ValueError:
        return 'Other'
    
    if 250 <= num < 251:
        return 'Diabetes'
    elif 390 <= num < 460:
        return 'Cardiovascular'
    elif 460 <= num < 520:
        return 'Respiratory'
    elif 520 <= num < 580:
        return 'Digestive'
    elif 580 <= num < 630:
        return 'Genitourinary'
    elif 140 <= num < 240:
        return 'Cancer'
    elif 800 <= num < 1000:
        return 'Injury'
    else:
        return 'Other'

# Apply to all three diagnosis columns
df['diag_1_group'] = df['diag_1'].apply(group_icd9)
df['diag_2_group'] = df['diag_2'].apply(group_icd9)
df['diag_3_group'] = df['diag_3'].apply(group_icd9)

# Verify distribution
print(df['diag_1_group'].value_counts())
print(df['diag_2_group'].value_counts())
print(df['diag_3_group'].value_counts())

diag_1_group
Cardiovascular    21322
Other             18565
Respiratory        6451
Digestive          6326
Diabetes           5748
Injury             4696
Genitourinary      3415
Cancer             2538
External            919
Unknown              10
Name: count, dtype: int64
diag_2_group
Cardiovascular    21787
Other             18801
Diabetes           9700
Respiratory        6448
Genitourinary      5044
Digestive          2706
Injury             1824
External           1787
Cancer             1600
Unknown             293
Name: count, dtype: int64
diag_3_group
Cardiovascular    20626
Other             19040
Diabetes          12547
Respiratory        4246
Genitourinary      3787
External           3517
Digestive          2447
Injury             1410
Unknown            1224
Cancer             1146
Name: count, dtype: int64


el historial de uso del sistema sanitario.
En el EDA viste que number_inpatient era el predictor más fuerte. Ahora vamos a crear una feature nueva que combine las tres en un solo número.

Por qué verificamos contra el target: siempre que creas una feature nueva, inmediatamente compruebas si discrimina entre las dos clases. Si la media es similar en ambos grupos, la feature no aporta nada al modelo.

Como interpretamos resultado:
No readmitidos:  0.53 contactos previos de media
Sí readmitidos:  0.83 contactos previos de media

Los pacientes que sí vuelven en menos de 30 días habían tenido, de media, un 55% más de contactos previos con el sistema sanitario que los que no vuelven.
Eso confirma que la variable nueva discrimina bien entre los dos grupos — tiene poder predictivo real. Si los dos números fueran parecidos, la variable no aportaría nada y la descartaríamos.

In [2]:
# Healthcare utilization — total prior contacts with the system
df['total_prior_contacts'] = (
    df['number_inpatient'] + 
    df['number_emergency'] + 
    df['number_outpatient']
)

# Verify — compare mean between groups
print(df.groupby('readmitted_30d')['total_prior_contacts'].mean())

readmitted_30d
0    0.533412
1    0.826730
Name: total_prior_contacts, dtype: float64


Fase 3 — Feature Engineering

Tomamos columnas que el modelo no puede usar bien
y las transformamos en variables con más poder predictivo.

diag_1/2/3 (900 códigos distintos)  →  9 grupos clínicos
number_inpatient + emergency + outpatient  →  total_prior_contacts
... más features que vienen a continuación

Solo 2.5 puntos de diferencia entre los dos grupos. La variable existe pero discrimina poco por sí sola.
¿Significa eso que la eliminamos? No necesariamente. Una variable débil individualmente puede aportar cuando el modelo la combina con otras. XGBoost es bueno encontrando esas interacciones. La mantenemos pero con expectativas bajas.
Esto también es un hallazgo honesto — no todas tus hipótesis se confirman fuerte. En tu README esto se escribe así: "medication change showed weak individual discriminative power (mean difference of 2.5pp between groups), though retained for potential interaction effects in the model."

In [3]:
# Medication change during hospitalization
df['medication_changed'] = (df['change'] == 'Ch').astype(int)

# Verify against target
print(df.groupby('readmitted_30d')['medication_changed'].mean())

readmitted_30d
0    0.447751
1    0.473031
Name: medication_changed, dtype: float64


Un paciente con 4 medicamentos de diabetes activos tiene una enfermedad más difícil de controlar que uno con 1. Es un proxy directo de la severidad de la diabetes

In [4]:
# Active diabetes medications count
diabetes_meds = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# Count medications that are active (not 'No')
df['num_diabetes_meds'] = (
    df[diabetes_meds] != 'No'
).sum(axis=1)

# Verify against target
print(df.groupby('readmitted_30d')['num_diabetes_meds'].mean())

readmitted_30d
0    1.186704
1    1.224344
Name: num_diabetes_meds, dtype: float64


No readmitidos:  1.19 medicamentos de diabetes activos
Sí readmitidos:  1.22 medicamentos de diabetes activos

In [5]:
# Convert age ranges to numeric midpoints
age_mapping = {
    '[0-10)'  : 5,
    '[10-20)' : 15,
    '[20-30)' : 25,
    '[30-40)' : 35,
    '[40-50)' : 45,
    '[50-60)' : 55,
    '[60-70)' : 65,
    '[70-80)' : 75,
    '[80-90)' : 85,
    '[90-100)': 95
}

df['age_numeric'] = df['age'].map(age_mapping)

# Verify against target
print(df.groupby('readmitted_30d')['age_numeric'].mean())

readmitted_30d
0    65.206577
1    67.836913
Name: age_numeric, dtype: float64


one hot enconding

In [6]:
# Columns to encode
categorical_cols = [
    'race', 'gender', 'age',
    'diag_1_group', 'diag_2_group', 'diag_3_group'
]

# One-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape before encoding: {df.shape}")
print(f"Shape after encoding:  {df_encoded.shape}")
print(f"New columns added: {df_encoded.shape[1] - df.shape[1]}")

Shape before encoding: (69990, 53)
Shape after encoding:  (69990, 89)
New columns added: 36


In [7]:
# Encode medication columns — No/Steady/Up/Down → numeric
med_mapping = {
    'No'    : 0,
    'Steady': 1,
    'Up'    : 2,
    'Down'  : 3
}

for col in diabetes_meds:
    df_encoded[col] = df_encoded[col].map(med_mapping)

# Encode diabetesMed — Yes/No → 1/0
df_encoded['diabetesMed'] = (df_encoded['diabetesMed'] == 'Yes').astype(int)

# Verify no object columns remain
obj_cols = df_encoded.select_dtypes(include='object').columns.tolist()
print(f"Remaining object columns: {obj_cols}")

Remaining object columns: ['diag_1', 'diag_2', 'diag_3', 'change', 'readmitted']


In [9]:
# Clean column names — remove characters not accepted by XGBoost
df_final.columns = (
    df_final.columns
    .str.replace('[', '', regex=False)
    .str.replace(']', '', regex=False)
    .str.replace('<', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(' ', '_', regex=False)
)

print("Columns cleaned.")
print(df_final.columns.tolist())

NameError: name 'df_final' is not defined

IMPORTANT:  This are the columns which the models predcition is based on:

In [ ]:
# Columns to drop — identifiers and replaced originals
cols_to_drop = [
    'encounter_id',      # identifier, no predictive value
    'patient_nbr',       # identifier, no predictive value
    'readmitted',        # original target, replaced by readmitted_30d
    'diag_1',            # replaced by diag_1_group (already encoded)
    'diag_2',            # replaced by diag_2_group (already encoded)
    'diag_3',            # replaced by diag_3_group (already encoded)
    'change',            # replaced by medication_changed
]

df_final = df_encoded.drop(columns=cols_to_drop)

print(f"Final dataset shape: {df_final.shape}")
print(f"\nTarget distribution:")
print(df_final['readmitted_30d'].value_counts(normalize=True).round(3))

# Save
df_final.to_csv('../data/processed/diabetic_data_features.csv', index=False)
print("\nSaved to data/processed/diabetic_data_features.csv")

Final dataset shape: (69990, 82)

Target distribution:
readmitted_30d
0    0.91
1    0.09
Name: proportion, dtype: float64

Saved to data/processed/diabetic_data_features.csv
